In [6]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft datasets transformers huggingface_hub
!pip install --no-deps trl peft accelerate bitsandbytes datasets scikit-learn pandas matplotlib

In [11]:
!nvidia-smi

Sun Jun  7 04:12:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
REPO_URL = "https://github.com/patibandlavenkatamanideep/relayops"

import os, shutil
%cd /content
if os.path.isdir("relayops"):
    shutil.rmtree("relayops")
!git clone $REPO_URL relayops
%cd /content/relayops

In [16]:
!python -m src.eval.export_finetune_data
print("\nsample line:")
!head -1 src/eval/data/finetune/train.jsonl

exported 180 examples to /content/relayops/relayops/relayops/src/eval/data/finetune
  train: 132
  val: 24
  test: 24

sample line:
{"messages": [{"role": "system", "content": "You are an intent classifier for a telecom customer-service agent. Classify the user's message into exactly one of these intents: reset_device, device_status, device_faq, billing, greeting, unknown. Respond with ONLY valid JSON of the form {\"intent\": \"<one_label>\"} and nothing else."}, {"role": "user", "content": "what's the best place to put my router"}, {"role": "assistant", "content": "{\"intent\": \"device_faq\"}"}]}


In [17]:
!python -m src.router.finetune_train

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 338/338 [00:01<00:00, 284.31it/s]
config.json: 100% 1.58k/1.58k [00:00<00:00, 3.66MB/s]
tokenizer_config.json: 100% 7.36k/7.36k [00:00<00:00, 14.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 78.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 74.6MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:01<00:00, 10.3MB/s]
added_tokens.json: 100% 605/605 [00:00<00:00, 3.06MB/s]
special_tokens_map.js

In [18]:
!RELAYOPS_INTENT_MODEL=models/intent-qwen2.5-1.5b-lora python -m src.eval.run_intent_eval

dataset: 180 examples  |  train: 126  test: 54

===== keyword baseline =====
        label  prec   rec    f1     support
 reset_device  1.00  0.22  0.36   9
device_status  0.88  0.78  0.82   9
   device_faq  0.88  0.78  0.82   9
      billing  1.00  0.33  0.50   9
     greeting  0.75  0.67  0.71   9
      unknown  0.32  0.89  0.47   9

accuracy: 0.611   macro-F1: 0.615

confusion matrix (rows=true):
                reset_device device_status    device_faq       billing      greeting       unknown
               (predicted ->)
 reset_device              2             1             0             0             0             6
device_status              0             7             0             0             1             1
   device_faq              0             0             7             0             0             2
      billing              0             0             1             3             0             5
     greeting              0             0             0             0  

In [ ]:
# Upload the LoRA adapter to the Hugging Face Hub (run in the SAME session that
# trained it; do NOT re-run the clone cell first — it deletes models/).
# Builds a CLEAN folder (adapter + tokenizer only, no training checkpoints) and
# attaches a model card, then makes the repo public.
import os, shutil
from getpass import getpass
from huggingface_hub import HfApi, login

SRC = "models/intent-qwen2.5-1.5b-lora"
CLEAN = "models/intent-qwen-clean"
assert os.path.isdir(SRC), f"{SRC} missing - re-run training (Colab runtimes are ephemeral)"

shutil.rmtree(CLEAN, ignore_errors=True)
os.makedirs(CLEAN, exist_ok=True)
KEEP = [
    "adapter_model.safetensors", "adapter_config.json",
    "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
    "chat_template.jinja", "vocab.json", "merges.txt", "added_tokens.json",
]
for f in KEEP:
    p = os.path.join(SRC, f)
    if os.path.exists(p):
        shutil.copy(p, CLEAN)
# model card -> Hub README
if os.path.exists("MODEL_CARD.md"):
    shutil.copy("MODEL_CARD.md", os.path.join(CLEAN, "README.md"))
print("uploading:", sorted(os.listdir(CLEAN)))

HF_TOKEN = getpass("HF write token: ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
user = api.whoami()["name"]                 # fails loudly if the token is invalid
HF_REPO = f"{user}/relayops-intent-qwen"    # your own namespace
api.create_repo(HF_REPO, repo_type="model", private=False, exist_ok=True)
try:
    api.update_repo_visibility(repo_id=HF_REPO, private=False)  # make public
except Exception as e:
    print(f"(set visibility manually in HF settings if needed: {e})")
api.upload_folder(folder_path=CLEAN, repo_id=HF_REPO, repo_type="model")
print("pushed clean ->", HF_REPO)

In [22]:
import shutil
shutil.make_archive("intent-lora", "zip", "models/intent-qwen2.5-1.5b-lora")
from google.colab import files
files.download("intent-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>